# Malilangwe Wildlife Detector — WAID Fine-Tuning

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Edit settings below if needed
3. `Runtime → Run all`

**Important:** Cell 2 mounts your Google Drive. Weights are saved there automatically after training so they survive disconnects.

## ⚙️ Settings

In [ ]:
# How many epochs this session (30 = ~50 min, safe for free Colab)
EPOCHS = 30

# Batch size (8 = safe for free T4)
BATCH = 8

# Resuming from a previous session?
# False = start fresh | True = loads weights from Google Drive automatically
RESUME = False

print(f'Settings: EPOCHS={EPOCHS}, BATCH={BATCH}, RESUME={RESUME}')

## 1. Mount Google Drive

Weights are saved here after training — they survive if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/malilangwe_weights'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Weights will be saved to: {DRIVE_DIR}')

## 2. Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR = '/content/wildlife-detector-malilangwe'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3. Download WAID Images

In [ ]:
WAID_DIR = '/content/wildlife-detector-malilangwe/WAID'

if not os.path.exists(os.path.join(WAID_DIR, 'WAID', 'images')):
    print('Downloading WAID dataset (~2 GB)...')
    !git clone --depth=1 https://github.com/xiaohuicui/WAID.git {WAID_DIR}/WAID_repo
    !cp -r {WAID_DIR}/WAID_repo/WAID/images {WAID_DIR}/WAID/images
    print('Done.')
else:
    print('WAID images already present.')

## 4. Validate Dataset & Generate YAML

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data.dataset import validate_dataset, get_class_distribution, generate_dataset_yaml
from src.utils.logging_setup import setup_logging

cfg = load_config()
setup_logging(cfg)

stats = validate_dataset(cfg)
print(f"Total images : {stats['total_images']:,}")
print(f"Total labels : {stats['total_labels']:,}")

print('\nClass distribution (train):')
for name, count in sorted(get_class_distribution(cfg, split='train').items(), key=lambda x: -x[1]):
    print(f'  {name:<12s} {count:>8,}')

dataset_yaml = generate_dataset_yaml(cfg)
print('\nDataset YAML:', dataset_yaml)

## 5. Train

Weights are copied to Google Drive automatically when done.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

train_cfg = cfg.training
det_cfg = cfg.detection

# Load model — fresh start or resume from Drive
if RESUME:
    ckpt = Path(DRIVE_DIR) / 'waid_last.pt'
    if not ckpt.exists():
        raise FileNotFoundError(f'No checkpoint found in Drive at {ckpt}. Set RESUME=False to start fresh.')
    print(f'Resuming from {ckpt}')
    model = YOLO(str(ckpt))
    resume_flag = True
else:
    print(f'Starting fresh from pretrained {det_cfg.model_variant}.pt')
    model = YOLO(f'{det_cfg.model_variant}.pt')
    resume_flag = False

results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=int(train_cfg.image_size),
    optimizer=str(train_cfg.optimizer),
    lr0=float(train_cfg.learning_rate),
    weight_decay=float(train_cfg.weight_decay),
    patience=int(train_cfg.patience),
    device=0,
    hsv_h=float(train_cfg.augmentation.hsv_h),
    hsv_s=float(train_cfg.augmentation.hsv_s),
    hsv_v=float(train_cfg.augmentation.hsv_v),
    flipud=float(train_cfg.augmentation.flipud),
    fliplr=float(train_cfg.augmentation.fliplr),
    mosaic=float(train_cfg.augmentation.mosaic),
    mixup=float(train_cfg.augmentation.mixup),
    scale=float(train_cfg.augmentation.scale),
    project='runs/train',
    name='waid_yolo11n',
    exist_ok=True,
    resume=resume_flag,
)

# Save weights to Google Drive immediately
w = Path('runs/train/waid_yolo11n/weights')
for fname in ['best.pt', 'last.pt']:
    src = w / fname
    if src.exists():
        dest = Path(DRIVE_DIR) / f'waid_{fname}'
        shutil.copy(src, dest)
        print(f'Saved {fname} to Drive: {dest}')

print('\nTraining complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

## 6. Evaluate

In [ ]:
from pathlib import Path
from ultralytics import YOLO

best = Path('runs/train/waid_yolo11n/weights/best.pt')
if not best.exists():
    # Try loading from Drive
    best = Path(DRIVE_DIR) / 'waid_best.pt'

if best.exists():
    eval_model = YOLO(str(best))
    metrics = eval_model.val(
        data=str(dataset_yaml), split='test',
        conf=float(det_cfg.confidence_threshold),
        iou=float(det_cfg.iou_threshold),
        imgsz=int(det_cfg.image_size),
        device=0, plots=True,
    )
    print(f'mAP@50:    {metrics.box.map50:.4f}')
    print(f'mAP@50-95: {metrics.box.map:.4f}')
    print(f'Precision:  {metrics.box.mp:.4f}')
    print(f'Recall:     {metrics.box.mr:.4f}')
else:
    print('No weights found — did training finish?')

## 7. Download Weights to Your Computer

Weights are already in your Google Drive (`My Drive/malilangwe_weights/`).  
This cell also triggers a direct browser download as a backup.

In [ ]:
from google.colab import files
from pathlib import Path

drive_weights = Path(DRIVE_DIR)
for fname in ['waid_best.pt', 'waid_last.pt']:
    p = drive_weights / fname
    if p.exists():
        files.download(str(p))
        print(f'Downloading {fname}...')
    else:
        print(f'{fname} not found in Drive')

print('\nDone!')
print('Put waid_best.pt in your local weights/ folder.')
print('Next session: set RESUME=True (loads waid_last.pt from Drive automatically).')

## 8. Test on a Custom Image (Optional)

In [ ]:
from google.colab import files as colab_files
from IPython.display import Image, display
from pathlib import Path
import glob

best = Path('runs/train/waid_yolo11n/weights/best.pt')
if not best.exists():
    best = Path(DRIVE_DIR) / 'waid_best.pt'

if not best.exists():
    print('No weights found. Run training first.')
else:
    print('Upload an aerial image...')
    uploaded = colab_files.upload()
    if uploaded:
        img_path = list(uploaded.keys())[0]
        m = YOLO(str(best))
        res = m.predict(source=img_path, conf=0.15, save=True,
                        project='/content/test_output', name='result', exist_ok=True)
        for r in res:
            print(f'Detections: {len(r.boxes)}')
            for box in r.boxes:
                print(f'  {r.names[int(box.cls[0])]}: {float(box.conf[0]):.0%}')
        saved = glob.glob('/content/test_output/result/*.jpg') + glob.glob('/content/test_output/result/*.png')
        if saved:
            display(Image(saved[0]))